# Lesson 03: Embeddings and hybrid search

**Goal:** embed chunks, index them in Qdrant and compare dense, BM25 and hybrid search.

**Before you start**
- Theory: [lesson 03](../../docs/lessons/03_embeddings_and_search.md)
- Python concepts: `Protocol`, a string-keyed registry, dict/sort with `key=`
- Files used: `fixtures/tiny.pdf`, the fake embedder, Qdrant in-memory. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import warnings
from pathlib import Path

from docrag.embed.base import EmbeddingService
from docrag.embed.registry import make_embedder
from docrag.ingest.pipeline import ingest_pdf
from docrag.store.qdrant import QdrantStore, make_client

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
chunks = ingest_pdf(ROOT / "fixtures" / "tiny.pdf", "tiny")
# In-memory Qdrant warns that payload indexes do nothing locally; expected here.
warnings.filterwarnings("ignore", message="Payload indexes have no effect")
print(len(chunks), "chunks:", [c.id for c in chunks])

## Step 1: Get an embedder from the registry

`make_embedder("<backend>:<model>", models_dir)` maps a spec string to an implementation (a tiny plugin registry, like a DI container keyed by name). `"fake:64"` is a deterministic hashing embedder with 64 dimensions: no model download. Anything with `name`, `version`, `dim` and `embed()` satisfies the `Embedder` Protocol.

In [ ]:
# TODO: the fake embedder with 64 dimensions (models_dir: ROOT / "models" / "hf")
embedder = None

In [ ]:
import numpy as np

assert embedder is not None and embedder.dim == 64, (
    "Hint: make_embedder('fake:64', ...)"
)
embeddings = EmbeddingService(embedder)  # adds batching, caching and L2 normalization
vectors = embeddings.embed([c.embed_text for c in chunks])
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0), (
    "EmbeddingService returns unit vectors"
)
store = QdrantStore(make_client(":memory:"), "lesson03", embeddings.dim)
store.ensure_collection()
store.upsert(chunks, vectors)
print(vectors.shape, "indexed:", store.count())
print("OK")

## Step 2: Dense vs. BM25 for a question with a number

`store.search(query_vector=..., query_text=..., mode=..., limit=...)` runs one retrieval mode. Dense needs a query vector (`embeddings.embed_query(q)`); BM25 only needs the text (`query_vector=None`). Run both for the question below.

> Honest caveat: the *fake* embedder hashes words, so it is lexical too, and here both modes agree. With a real embedding model the two diverge on numbers and IDs (see the observation question).

In [ ]:
q = "Which region had 45.5 million euros?"
# TODO: dense = store.search(... mode="dense", limit=3)   (needs embeddings.embed_query(q))
# TODO: bm25  = store.search(... mode="bm25",  limit=3)   (query_vector=None)
dense = []
bm25 = []

In [ ]:
assert dense and bm25, (
    "Hint: store.search(query_vector=..., query_text=q, mode='dense', limit=3)"
)
assert bm25[0].chunk.id == "tiny:p2:c1", (
    "BM25 should rank the table (which contains '45.5') first"
)
for name, hits in [("dense", dense), ("bm25", bm25)]:
    print(name.ljust(6), [(h.chunk.id, round(h.score, 3)) for h in hits])
print("OK")

## Step 3: Reciprocal Rank Fusion by hand

Hybrid search fuses the two ranked lists with **RRF**: `score(d) = sum over lists of 1 / (k + rank)`, with rank starting at 1 and `k = 60`. It only uses ranks, so cosine scores (0..1) and BM25 scores (0..30+) never need calibrating against each other. Implement it and return `(id, score)` pairs, best first.

In [ ]:
def rrf(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    # TODO: accumulate 1 / (k + rank) per id over all lists (enumerate(..., start=1)),
    # then return sorted(scores.items(), key=..., reverse=True)
    return []

In [ ]:
fused = rrf([["a", "b", "c"], ["b", "c", "d"]])
assert fused, "rrf() returned nothing"
assert fused[0][0] == "b", "'b' is high in BOTH lists, so it should win"
assert abs(dict(fused)["b"] - (1 / 62 + 1 / 61)) < 1e-12, (
    "b: rank 2 in list 1, rank 1 in list 2"
)
print([(d, round(s, 5)) for d, s in fused])
hybrid = store.search(
    query_vector=embeddings.embed_query(q), query_text=q, mode="hybrid", limit=3
)
print("Qdrant hybrid:", [h.chunk.id for h in hybrid])
print("OK")

## Observation question

Why does BM25 find numbers and IDs better than dense search?

<details>
<summary>Answer</summary>

BM25 matches exact tokens, and `tokenize` keeps `45.5` as one token; a rare token has a high IDF, so chunks containing it get a big boost. (In `tiny.pdf` the text chunk on page 2 also says 45.5, which is why the two BM25 scores are close.) A real dense embedding compresses a passage into a meaning vector: numbers, codes and IDs carry little "meaning", look alike, and models split them into sub-word pieces. The fake embedder used here doesn't show this because it is a bag of words itself. That's why the project uses hybrid search (dense for paraphrases, BM25 for exact terms), fused with RRF.
</details>

## The real code

- Embedder + service: [`src/docrag/embed/base.py`](../../src/docrag/embed/base.py)
- Registry: [`src/docrag/embed/registry.py`](../../src/docrag/embed/registry.py)
- BM25 sparse vectors: [`src/docrag/store/bm25.py`](../../src/docrag/store/bm25.py)
- Qdrant store + hybrid: [`src/docrag/store/qdrant.py`](../../src/docrag/store/qdrant.py)
- Retriever: [`src/docrag/store/retriever.py`](../../src/docrag/store/retriever.py)
- Lesson: [`docs/lessons/03_embeddings_and_search.md`](../../docs/lessons/03_embeddings_and_search.md)